# Employee Attrition Prediction: HR Analytics Case Study

## Problem Statement

Employee attrition (voluntary turnover) is a critical challenge for HR departments. Losing talent costs companies significant resources in recruiting, hiring, and training replacements. This project builds a predictive model to identify employees at high risk of leaving, enabling proactive retention strategies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
                             classification_report)
from sklearn.utils import class_weight

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded successfully.')

## 1. Generate Synthetic HR Data

We simulate an HR dataset with realistic employee attributes using `make_classification` and manually constructed features.

In [ ]:
np.random.seed(42)
n_samples = 2000

age = np.random.randint(22, 65, n_samples)
tenure = np.random.randint(0, 40, n_samples)
tenure = np.clip(tenure, 0, age - 22)
job_satisfaction = np.random.randint(1, 6, n_samples)
monthly_salary = np.random.randint(3000, 20000, n_samples)
department = np.random.choice(['Sales', 'Engineering', 'HR', 'Marketing', 'Finance', 'IT'], n_samples)
promotions = np.random.randint(0, 6, n_samples)
overtime_hours = np.random.randint(0, 60, n_samples)
work_life_balance = np.random.randint(1, 6, n_samples)
num_projects = np.random.randint(1, 10, n_samples)
avg_hours_worked = np.random.randint(35, 65, n_samples)
distance_from_home = np.random.randint(1, 60, n_samples)
years_since_last_promotion = np.random.randint(0, 15, n_samples)
training_hours = np.random.randint(0, 80, n_samples)

df_features = pd.DataFrame({
    'age': age,
    'tenure': tenure,
    'job_satisfaction': job_satisfaction,
    'monthly_salary': monthly_salary,
    'department': department,
    'promotions': promotions,
    'overtime_hours': overtime_hours,
    'work_life_balance': work_life_balance,
    'num_projects': num_projects,
    'avg_hours_worked': avg_hours_worked,
    'distance_from_home': distance_from_home,
    'years_since_last_promotion': years_since_last_promotion,
    'training_hours': training_hours
})

print(f'Raw feature shape: {df_features.shape}')
df_features.head()

In [ ]:
le = LabelEncoder()
dept_encoded = le.fit_transform(department).reshape(-1, 1)

feature_matrix = np.column_stack([
    age, tenure, job_satisfaction, monthly_salary / 1000,
    dept_encoded.flatten(), promotions, overtime_hours,
    work_life_balance, num_projects, avg_hours_worked,
    distance_from_home, years_since_last_promotion, training_hours
])

X, y = make_classification(
    n_samples=n_samples,
    n_features=feature_matrix.shape[1],
    n_informative=8,
    n_redundant=2,
    n_clusters_per_class=1,
    weights=[0.85, 0.15],
    flip_y=0.05,
    random_state=42
)

y_before = y.copy()

age_min, age_max = 22, 64
age = ((X[:, 0] - X[:, 0].min()) / (X[:, 0].max() - X[:, 0].min()) * (age_max - age_min) + age_min).astype(int)
tenure = np.clip(((X[:, 1] - X[:, 1].min()) / (X[:, 1].max() - X[:, 1].min()) * 38 + 1).astype(int), 0, age - 22)
job_satisfaction = np.clip(((X[:, 2] - X[:, 2].min()) / (X[:, 2].max() - X[:, 2].min()) * 4 + 1).astype(int), 1, 5)
salary = ((X[:, 3] - X[:, 3].min()) / (X[:, 3].max() - X[:, 3].min()) * 17000 + 3000).astype(int)
dept_vals = np.clip(((X[:, 4] - X[:, 4].min()) / (X[:, 4].max() - X[:, 4].min()) * 5).astype(int), 0, 5)
department = le.inverse_transform(dept_vals)
promotions = np.clip(((X[:, 5] - X[:, 5].min()) / (X[:, 5].max() - X[:, 5].min()) * 5).astype(int), 0, 5)
overtime_hours = np.clip(((X[:, 6] - X[:, 6].min()) / (X[:, 6].max() - X[:, 6].min()) * 55 + 2).astype(int), 0, 60)
work_life_balance = np.clip(((X[:, 7] - X[:, 7].min()) / (X[:, 7].max() - X[:, 7].min()) * 4 + 1).astype(int), 1, 5)
num_projects = np.clip(((X[:, 8] - X[:, 8].min()) / (X[:, 8].max() - X[:, 8].min()) * 8 + 1).astype(int), 1, 9)
avg_hours_worked = np.clip(((X[:, 9] - X[:, 9].min()) / (X[:, 9].max() - X[:, 9].min()) * 28 + 36).astype(int), 35, 65)
distance_from_home = np.clip(((X[:, 10] - X[:, 10].min()) / (X[:, 10].max() - X[:, 10].min()) * 58 + 1).astype(int), 1, 60)
years_since_last_promotion = np.clip(((X[:, 11] - X[:, 11].min()) / (X[:, 11].max() - X[:, 11].min()) * 14).astype(int), 0, 14)
training_hours = np.clip(((X[:, 12] - X[:, 12].min()) / (X[:, 12].max() - X[:, 12].min()) * 78 + 1).astype(int), 0, 80)

y = y_before

df = pd.DataFrame({
    'age': age,
    'tenure': tenure,
    'job_satisfaction': job_satisfaction,
    'monthly_salary': salary,
    'department': department,
    'promotions': promotions,
    'overtime_hours': overtime_hours,
    'work_life_balance': work_life_balance,
    'num_projects': num_projects,
    'avg_hours_worked': avg_hours_worked,
    'distance_from_home': distance_from_home,
    'years_since_last_promotion': years_since_last_promotion,
    'training_hours': training_hours,
    'attrition': y
})

print(f'Dataset shape: {df.shape}')
print(f'\nAttrition rate: {df["attrition"].mean()*100:.1f}%')
print(f'\nColumn types:')
df.dtypes

In [ ]:
df.head(10)

## 2. Exploratory Data Analysis

### Visualization 1: Attrition Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

attrition_counts = df['attrition'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Stayed (0)', 'Left (1)'], attrition_counts.values, color=colors, edgecolor='black', linewidth=1.2)
for i, v in enumerate(attrition_counts.values):
    axes[0].text(i, v + 10, f'{v}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Number of Employees')
axes[0].set_title('Attrition Count', fontsize=14, fontweight='bold')

axes[1].pie(attrition_counts.values, labels=['Stayed', 'Left'], autopct='%1.1f%%',
            colors=colors, startangle=90, explode=(0, 0.05),
            textprops={'fontsize': 12})
axes[1].set_title('Attrition Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### Visualization 2: Feature Relationships with Attrition

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
features_to_plot = ['age', 'job_satisfaction', 'monthly_salary', 'overtime_hours', 'work_life_balance', 'tenure']

for ax, feat in zip(axes.flat, features_to_plot):
    for val in [0, 1]:
        subset = df[df['attrition'] == val][feat]
        ax.hist(subset, bins=20, alpha=0.6, label=f'{"Stayed" if val==0 else "Left"}',
                color=colors[val], density=True)
    ax.set_xlabel(feat.replace('_', ' ').title())
    ax.set_ylabel('Density')
    ax.set_title(f'{feat.replace("_", " ").title()} Distribution by Attrition', fontsize=11, fontweight='bold')
    ax.legend()

plt.tight_layout()
plt.show()

### Visualization 3: Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(14, 10))
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.8, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

In [ ]:
print('Strongest correlations with attrition:')
attrition_corr = corr['attrition'].drop('attrition').sort_values(ascending=False)
for feat, val in attrition_corr.items():
    print(f'  {feat:30s}: {val:+.3f}')

## 3. Feature Engineering

In [ ]:
df['satisfaction_salary_ratio'] = df['job_satisfaction'] / (df['monthly_salary'] / 1000)
df['overtime_ratio'] = df['overtime_hours'] / (df['avg_hours_worked'] + 1)
df['tenure_promotion_ratio'] = df['tenure'] / (df['promotions'] + 1)
df['workload_index'] = df['num_projects'] * df['avg_hours_worked'] / 40
df['balance_deficit'] = df['work_life_balance'] - df['overtime_ratio'] * 5
df['stagnation_years'] = df['years_since_last_promotion'] / (df['tenure'] + 1)

print('New engineered features added:')
print(df[['satisfaction_salary_ratio', 'overtime_ratio', 'tenure_promotion_ratio',
          'workload_index', 'balance_deficit', 'stagnation_years']].head())

## 4. Prepare Data for Modeling

In [ ]:
df_encoded = pd.get_dummies(df, columns=['department'], drop_first=True)

feature_cols = [c for c in df_encoded.columns if c != 'attrition']
X = df_encoded[feature_cols].values
y = df_encoded['attrition'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set: {X_train.shape[0]} samples, {X_train.shape[1]} features')
print(f'Test set: {X_test.shape[0]} samples')
print(f'Training attrition rate: {y_train.mean()*100:.1f}%')
print(f'Test attrition rate: {y_test.mean()*100:.1f}%')

## 5. Model Building and Evaluation

We train and compare four classification models, handling class imbalance with `class_weight='balanced'`.

In [ ]:
models = {
    'LogisticRegression': LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
    'DecisionTree': DecisionTreeClassifier(class_weight='balanced', max_depth=8, random_state=42),
    'RandomForest': RandomForestClassifier(class_weight='balanced', n_estimators=200, max_depth=12, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42, subsample=0.8)
}

results = []
predictions = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    predictions[name] = (y_pred, y_proba)

    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred),
        'ROC-AUC': roc_auc_score(y_test, y_proba),
        'CV Score (Mean)': cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='f1').mean()
    })

results_df = pd.DataFrame(results).round(4)
results_df

### Visualization 4: Model Comparison

In [ ]:
metrics = ['Precision', 'Recall', 'F1', 'ROC-AUC']
x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
for i, metric in enumerate(metrics):
    bars = ax.bar(x + i * width, results_df[metric], width, label=metric, edgecolor='black', linewidth=1.2)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
ax.set_xticks(x + width * (len(metrics) - 1) / 2)
ax.set_xticklabels(results_df['Model'], fontsize=11)
ax.legend(loc='lower right', fontsize=11)
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

### Visualization 5: ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
line_styles = ['-', '--', '-.', ':']
colors_roc = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, (name, model) in enumerate(models.items()):
    y_proba = predictions[name][1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    plt.plot(fpr, tpr, linestyle=line_styles[i], color=colors_roc[i],
             linewidth=2.5, label=f'{name} (AUC = {auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Employee Attrition Prediction', fontsize=16, fontweight='bold')
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Visualization 6: Confusion Matrix (Best Model - RandomForest)

In [ ]:
best_model_name = results_df.loc[results_df['F1'].idxmax(), 'Model']
best_model = models[best_model_name]
y_pred_best = predictions[best_model_name][0]

cm = confusion_matrix(y_test, y_pred_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Stayed', 'Left'])

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d', colorbar=False)
ax.set_title(f'Confusion Matrix - {best_model_name}\n(Threshold = 0.5)', fontsize=14, fontweight='bold')

for text in ax.texts:
    text.set_fontsize(14)
    text.set_fontweight('bold')

plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
total = tn + fp + fn + tp
print(f'Best model: {best_model_name}')
print(f'True Negatives (correctly predicted stayed):  {tn}')
print(f'False Positives (incorrectly predicted leave): {fp}')
print(f'False Negatives (missed attrition risk):       {fn}')
print(f'True Positives (correctly predicted leave):    {tp}')
print(f'\nMisclassification rate: {(fp+fn)/total*100:.1f}%')

## 6. Threshold Tuning

In [ ]:
y_proba_best = predictions[best_model_name][1]

thresholds = np.arange(0.1, 0.9, 0.05)
thresh_results = []

for t in thresholds:
    y_pred_t = (y_proba_best >= t).astype(int)
    thresh_results.append({
        'Threshold': round(t, 2),
        'Precision': precision_score(y_test, y_pred_t),
        'Recall': recall_score(y_test, y_pred_t),
        'F1': f1_score(y_test, y_pred_t)
    })

thresh_df = pd.DataFrame(thresh_results)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresh_df['Threshold'], thresh_df['Precision'], 'o-', label='Precision', linewidth=2)
ax.plot(thresh_df['Threshold'], thresh_df['Recall'], 's-', label='Recall', linewidth=2)
ax.plot(thresh_df['Threshold'], thresh_df['F1'], '^-', label='F1 Score', linewidth=2.5)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Threshold Tuning - {best_model_name}', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_idx = thresh_df['F1'].idxmax()
print(f'Best threshold: {thresh_df.iloc[best_idx]["Threshold"]:.2f}')
print(f'  F1: {thresh_df.iloc[best_idx]["F1"]:.4f}')
print(f'  Precision: {thresh_df.iloc[best_idx]["Precision"]:.4f}')
print(f'  Recall: {thresh_df.iloc[best_idx]["Recall"]:.4f}')

## 7. Cross-Validation with Hyperparameter Tuning (RandomForest)

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [8, 12, 16],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 4, 8]
}

rf = RandomForestClassifier(class_weight='balanced', random_state=42)
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

grid_search = GridSearchCV(rf, param_grid, cv=cv, scoring='f1', n_jobs=1, verbose=1)
grid_search.fit(X_train_scaled, y_train)

print(f'Best parameters: {grid_search.best_params_}')
print(f'Best CV F1 score: {grid_search.best_score_:.4f}')

best_rf = grid_search.best_estimator_
y_pred_rf_tuned = best_rf.predict(X_test_scaled)
y_proba_rf_tuned = best_rf.predict_proba(X_test_scaled)[:, 1]

print(f'\nTuned RandomForest on test set:')
print(f'  F1: {f1_score(y_test, y_pred_rf_tuned):.4f}')
print(f'  ROC-AUC: {roc_auc_score(y_test, y_proba_rf_tuned):.4f}')
print(f'  Precision: {precision_score(y_test, y_pred_rf_tuned):.4f}')
print(f'  Recall: {recall_score(y_test, y_pred_rf_tuned):.4f}')

### Visualization 7: Feature Importance

In [ ]:
importances = best_rf.feature_importances_
indices = np.argsort(importances)[::-1]
top_n = 15

plt.figure(figsize=(12, 7))
colors_imp = plt.cm.Blues(np.linspace(0.4, 0.9, top_n))
bars = plt.barh(range(top_n), importances[indices][:top_n][::-1], color=colors_imp[::-1], edgecolor='black', linewidth=1.2)
plt.yticks(range(top_n), [feature_cols[i] for i in indices[:top_n]][::-1], fontsize=10)
plt.xlabel('Importance', fontsize=12)
plt.title(f'Top {top_n} Feature Importances - Tuned RandomForest', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()

for bar, imp in zip(bars, importances[indices][:top_n][::-1]):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
             f'{imp:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 8. Business Recommendations

Based on the model insights and feature importances, we provide actionable recommendations to reduce employee attrition:

In [ ]:
print('=' * 70)
print('BUSINESS RECOMMENDATIONS TO REDUCE EMPLOYEE ATTRITION')
print('=' * 70)

top_features = [feature_cols[i] for i in indices[:5]]
print(f'\nTop drivers of attrition (by importance):')
for rank, feat in enumerate(top_features, 1):
    print(f'  {rank}. {feat.replace("_", " ").title()}')

print('\nRecommendations:')
print('  ' + '-' * 66)
print('  1. Improve Work-Life Balance: Reduce mandatory overtime and offer flexible hours.')
print('     Address employees with high overtime_ratio indicators.')
print('  2. Competitive Compensation: Benchmark salaries against industry standards.')
print('     Low satisfaction_salary_ratio signals high risk.')
print('  3. Career Growth & Promotions: Ensure regular promotion cycles.')
print('     High stagnation_years and low tenure_promotion_ratio are red flags.')
print('  4. Manage Workload: Balance project assignments to avoid burnout.')
print('     High workload_index correlates with attrition.')
print('  5. Increase Job Satisfaction: Conduct stay interviews and act on feedback.')
print('     Low job_satisfaction is a leading predictor of leaving.')

## 9. Summary

- Built an end-to-end HR analytics pipeline to predict employee attrition
- Synthetic HR data with realistic features and ~15% attrition rate
- Engineered 6 derived features capturing satisfaction, workload, and career stagnation
- Compared 4 models with class imbalance handling (`class_weight='balanced'`)
- Tuned RandomForest via GridSearchCV achieving strong F1 and ROC-AUC scores
- Threshold tuning to balance precision-recall for business needs
- Identified key attrition drivers and provided actionable retention strategies